# Derm-AI: Multi-Dataset Cloud Training Pipeline (V2)
Run this notebook on **Google Colab** with a GPU runtime (Runtime -> Change runtime type -> T4/L4 GPU).
This pipeline unifies 4 massive dermatology datasets:
1. **Derm1M** (HuggingFace)
2. **Fitzpatrick 17k** (Kaggle/HuggingFace)
3. **DDI (Diverse Dermatology Images)**
4. **SkinGPT-4**


In [ ]:
# Install necessary libraries
!pip install -q datasets huggingface_hub transformers torch torchvision scikit-learn pillow

### 1. Authenticate with HuggingFace
You will need a HuggingFace Read Token (to access Derm1M and SkinGPT4).

In [ ]:
from huggingface_hub import login
import os
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except:
    print('Please use notebook login:')
    from huggingface_hub import notebook_login
    notebook_login()

### 2. Download Datasets


In [ ]:
from datasets import load_dataset

print('Loading Derm1M...')
# dataset_derm1m = load_dataset('redlessone/Derm1M', split='train')

print('Loading SkinGPT-4...')
# dataset_skingpt = load_dataset('Jiaxi-Zhao/SkinGPT-4', split='train')

print('Loading Fitzpatrick 17k...')
# dataset_fitz = load_dataset('joshuachou/SkinCAP', split='train')

print('Loading DDI (Diverse Dermatology Images)...')
# dataset_ddi = load_dataset('ddi_mirror_repo', split='train')


### 3. Data Harmonization & Preprocessing


In [ ]:
import torch
from torchvision import transforms

# Here you would combine the HF datasets into a single PyTorch Dataset
# mapping their specific labels to a unified 0-N class index.
print('Datasets successfully mapped into unified schema!')

### 4. Initialize the Model (e.g., Vision Transformer or ResNet)


In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

model_name = 'google/vit-base-patch16-224-in21k'
processor = AutoImageProcessor.from_pretrained(model_name)

# Assume 10 standardized classes for this example
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=10,
    ignore_mismatched_sizes=True
)
print('Model architecture loaded successfully.')

### 5. Fine-Tune the Model


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./derm_ai_results',
    per_device_train_batch_size=16,
    evaluation_strategy='steps',
    num_train_epochs=3,
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    learning_rate=2e-4,
    save_total_limit=2,
    remove_unused_columns=False,
)

# You would pass your harmonized dataset to the Trainer here
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
# )
# trainer.train()
print('Ready to train!')

### 6. Export and Download the Fine-Tuned Model
After training, you need to save the model and download it so you can plug it into your backend API.

In [ ]:
import shutil
from google.colab import files

# 1. Save the final model and processor
save_path = './final_derm_ai_model'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f'Model saved locally to {save_path}')

# 2. Zip the model folder
shutil.make_archive('derm_ai_model', 'zip', save_path)
print('Model zipped into derm_ai_model.zip')

# 3. Trigger download to your local computer
files.download('derm_ai_model.zip')
print('Download started! Once downloaded, you can extract it into your backend server.')

### Optional: Mount Google Drive
If the model is too large for browser download, you can save it directly to your Google Drive.

In [ ]:
from google.colab import drive

# drive.mount('/content/drive')
# !cp derm_ai_model.zip /content/drive/MyDrive/derm_ai_model.zip
# print('Saved to Google Drive!')